<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/deep_sarsa_live_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade --force-reinstall \
    dask==2024.11.2 \
    rapids-dask-dependency==24.12.0 \
    cudf-cu12==24.12.0 \
    cuml-cu12==24.12.0 \
    pylibraft-cu12==24.12.0 \
    pylibcudf-cu12==24.12.0 \
    numba==0.61.0 --quiet

In [2]:

#Install Stable Baselines3 and Trading Libraries
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib


In [3]:
#Clean install of TensorFlow compatible with Colab's GPU (CUDA 11.8 + cuDNN 8.x)
!pip install tensorflow==2.12.0

#Restart runtime after this!
!pip install numpy==1.24.4 --force-reinstall

In [1]:
import torch
import cudf
import cuml
import dask
import pandas as pd
import numpy as np
import scipy
import lightgbm as lgb
import gymnasium as gym
import stable_baselines3

#Version Checks
print(" Library Versions")
print("--------------------")
print(" PyTorch:", torch.__version__)
print(" CUDA:", torch.version.cuda)
print(" cuDF:", cudf.__version__)
print(" cuML:", cuml.__version__)
print(" Dask:", dask.__version__)
print(" Pandas:", pd.__version__)
print(" NumPy:", np.__version__)
print(" SciPy:", scipy.__version__)
print(" LightGBM:", lgb.__version__)
print(" Gymnasium:", gym.__version__)
print(" Stable Baselines3:", stable_baselines3.__version__)

# GPU Check (Torch + NVIDIA)
print("\n GPU Availability")
print("--------------------")
print(" PyTorch GPU Available:", torch.cuda.is_available())
print(" GPU Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print(" GPU Name:", torch.cuda.get_device_name(0))

In [2]:
#Clean install of TensorFlow compatible with Colab's GPU (CUDA 11.8 + cuDNN 8.x)
!pip uninstall -y tensorflow keras
!pip install tensorflow==2.12.0

#Restart runtime after this!
!pip install numpy==1.24.4 --force-reinstall

In [1]:
#Core & System Utilities
import os
import gc
import sys
import time
import json
import pickle
import random
from datetime import datetime
from collections import defaultdict, deque

#Data Science Essentials
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numba

#Financial Data
import yfinance as yf

#Machine Learning & Preprocessing
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

#Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import Input, backend as K
from tensorflow.keras import mixed_precision

#Visualization & Display
import IPython.display as display

#RAPIDS Libraries (for GPU-accelerated ML, optional)
import cupy as cp

#Reinforcement Learning (Stable Baselines3)
import stable_baselines3
from stable_baselines3 import A2C, DDPG, DQN, PPO, SAC, TD3
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.logger import configure
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

#Gym & Trading Environments
import gym
import gymnasium as gym
import gym_anytrading
from gym.spaces import Box
from gymnasium.spaces import Box as GymBox, Discrete
from gymnasium.wrappers import TimeLimit

#CUDA (Optional Paths - for manual GPU configuration)
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-11.8/lib64'

#GPU Check (Colab only)
!nvidia-smi


In [2]:
# Required for TensorFlow compatibility (GPU + cuDNN)
!pip uninstall -y tensorflow keras -q
!pip install tensorflow==2.12.0 -q



In [3]:
# Fix protobuf compatibility
!pip install protobuf==3.20.3 -q

# Essential packages
!pip install numpy==1.24.4 pandas joblib yfinance scikit-learn matplotlib -q

In [5]:
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("ogle Drive is already mounted.")



In [6]:
!rm -rf /content/drive

rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.file-revisions-by-id': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled


In [7]:
#Downgrade NumPy to a compatible version
!pip install numpy==1.24.4 --force-reinstall

#Reinstall LightGBM after fixing NumPy
!pip install lightgbm --force-reinstall --no-cache-dir


In [8]:
!pip install -U scikit-learn==1.3.2 --quiet


In [1]:
!pip install yfinance

In [5]:
# === Deep SARSA Walkforward with Full Model Saving + Selector Support ===
import os, gc, random, json
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime, timedelta

# Config
TICKERS = [
    "AAPL", "TSLA", "MSFT", "GOOG", "AMZN", "NVDA", "META", "JPM", "BAC", "WMT",
    "UNH", "V", "PG", "HD", "MA", "DIS", "PEP", "KO", "CSCO", "ADBE", "CRM", "NFLX",
    "PFE", "MRK", "T", "ORCL", "ABBV", "CVX", "XOM", "ABT", "COST", "QCOM", "INTC",
    "MCD", "NKE", "DHR", "LLY", "MDT", "TMO", "TXN", "PM", "AVGO", "NEE", "ACN", "UPS",
    "HON", "LIN", "GS", "IBM"
]  # Reduce for debug
SEQUENCE_LENGTH = 60
SAVE_DIR = "/content/drive/MyDrive/Results_May_2025/results_deep_sarsa_walkforward/deep_sarsa_walkforward_models"
RESULTS_DIR = "/content/drive/MyDrive/Results_May_2025/results_deep_sarsa_walkforward"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR + "/plots", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Feature Engineering
def compute_technical_indicators(df):
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20).mean()
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / (loss + 1e-6)
    df['RSI'] = 100 - (100 / (1 + rs))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['Signal_Line'] = df['MACD'].ewm(span=9).mean()
    df['ATR'] = df['High'].rolling(14).max() - df['Low'].rolling(14).min()
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    df.dropna(inplace=True)
    return df

# Deep SARSA Model
class DQNNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 64, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.out = nn.Linear(32, output_dim)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = h[:, -1, :]
        x = torch.relu(self.fc1(h))
        return self.out(x)

class DeepSARSAAgent:
    def __init__(self, input_dim, action_dim, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.model = DQNNet(input_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, 2)
        with torch.no_grad():
            q_values = self.model(state)
        return torch.argmax(q_values).item()

    def update(self, state, action, reward, next_state, next_action, done):
        q_values = self.model(state)
        next_q_values = self.model(next_state)
        target = reward + self.gamma * next_q_values[0, next_action] * (1 - int(done))
        loss = self.criterion(q_values[0, action], target.detach())
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

# Ensure maximum 729 days (safe range)
test_end_dt = datetime.today()
train_start_dt = test_end_dt - timedelta(days=729)

# Format
train_start = train_start_dt.strftime("%Y-%m-%d")
test_end = test_end_dt.strftime("%Y-%m-%d")
train_end = (train_start_dt + timedelta(days=365)).strftime("%Y-%m-%d")
test_start = train_end


results = []
for ticker in TICKERS:
    print(f"\n Training {ticker}")
    df = yf.download(ticker, start=train_start, end=test_end, interval="1h", progress=False)
    if df.empty or len(df) < SEQUENCE_LENGTH + 2:
        print(f" Skipping {ticker}: Insufficient data.")
        continue

    df = compute_technical_indicators(df)
    features = ['Close', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'Signal_Line', 'ATR', 'OBV']
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[features])
    df_scaled = pd.DataFrame(scaled, index=df.index, columns=features)

    sequences = [(df_scaled.index[i], df_scaled.iloc[i - SEQUENCE_LENGTH:i].values)
                 for i in range(SEQUENCE_LENGTH, len(df_scaled))]
    train_X = np.array([x[1] for x in sequences if train_start <= x[0].strftime('%Y-%m-%d') < train_end])
    test_X = np.array([x[1] for x in sequences if test_start <= x[0].strftime('%Y-%m-%d') < test_end])

    if len(train_X) < 10 or len(test_X) < 10:
        print(f" Skipping {ticker}: Not enough training/testing data.")
        continue

    agent = DeepSARSAAgent(train_X.shape[2], 3)
    for ep in range(5):
        idx = random.randint(0, len(train_X) - 2)
        state = torch.tensor(train_X[idx][None], dtype=torch.float32).to(device)
        action = agent.select_action(state, epsilon=1.0 - ep / 5)
        for t in range(idx, len(train_X) - 1):
            next_state = torch.tensor(train_X[t+1][None], dtype=torch.float32).to(device)
            reward = float(train_X[t+1, -1, 0] - train_X[t, -1, 0])
            done = t + 2 == len(train_X)
            next_action = agent.select_action(next_state, epsilon=1.0 - ep / 5)
            agent.update(state, action, reward, next_state, next_action, done)
            if done: break
            state, action = next_state, next_action

    # Evaluation
    actions, returns, labels = [], [], []
    for t in range(len(test_X) - 1):
        state = torch.tensor(test_X[t][None], dtype=torch.float32).to(device)
        action = agent.select_action(state, epsilon=0.0)
        actions.append(action)
        current_price = test_X[t, -1, 0]
        next_price = test_X[t+1, -1, 0]
        returns.append((next_price - current_price) / (current_price + 1e-6))
        labels.append(2 if next_price > current_price else 0 if next_price < current_price else 1)

    cumulative = np.cumprod(1 + np.array(returns))
    final_value = 100000 * cumulative[-1]
    sharpe = np.mean(returns) / (np.std(returns) + 1e-6) * np.sqrt(252)
    drawdown = np.max(np.maximum.accumulate(cumulative) - cumulative)
    accuracy = np.mean(np.array(actions) == np.array(labels))
    return_pct = cumulative[-1] - 1

    # Save model, scaler, features
    model_dir = os.path.join(SAVE_DIR, ticker)
    os.makedirs(model_dir, exist_ok=True)
    torch.save(agent.model.state_dict(), os.path.join(model_dir, "model.pth"))
    with open(os.path.join(model_dir, "features.json"), "w") as f:
        json.dump(features, f)
    import joblib
    joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(100000 * cumulative, label='Deep SARSA Strategy')
    plt.axhline(100000, linestyle='--', color='gray')
    plt.title(f"{ticker} - Deep SARSA Portfolio")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "plots", f"{ticker}_sarsa_portfolio.png"))
    plt.close()

    results.append({
        "Ticker": ticker,
        "Model": "DeepSARSA",
        "Sharpe": round(sharpe, 4),
        "Accuracy": round(accuracy, 4),
        "Drawdown": round(drawdown * 100, 2),
        "Return": round(return_pct * 100, 2),
        "Final_Portfolio": round(final_value, 2)
    })

    gc.collect()
    torch.cuda.empty_cache()

# Save All Metrics
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_DIR, "deep_sarsa_model_selector_metrics.csv"), index=False)

if not results_df.empty:
    results_df['score'] = (
        results_df['Sharpe'] * 0.4 +
        results_df['Return'] * 0.3 +
        results_df['Final_Portfolio'] * 0.3
    )
    results_df.to_csv(os.path.join(RESULTS_DIR, "deep_sarsa_model_selector_scored.csv"), index=False)
    best = results_df.sort_values(['Ticker', 'score'], ascending=[True, False]).groupby('Ticker').first().reset_index()
    best.to_excel(os.path.join(RESULTS_DIR, "deep_sarsa_best_models_by_score.xlsx"), index=False)
else:
    print(" No results to score.")



 Training AAPL

 Training TSLA

 Training MSFT

 Training GOOG

 Training AMZN

 Training NVDA

 Training META

 Training JPM

 Training BAC

 Training WMT

 Training UNH

 Training V

 Training PG

 Training HD

 Training MA

 Training DIS

 Training PEP

 Training KO

 Training CSCO

 Training ADBE

 Training CRM

 Training NFLX

 Training PFE

 Training MRK

 Training T

 Training ORCL

 Training ABBV

 Training CVX

 Training XOM

 Training ABT

 Training COST

 Training QCOM

 Training INTC

 Training MCD

 Training NKE

 Training DHR

 Training LLY

 Training MDT

 Training TMO

 Training TXN

 Training PM

 Training AVGO

 Training NEE

 Training ACN

 Training UPS

 Training HON

 Training LIN

 Training GS

 Training IBM
